In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [2]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: openpyxl in C:\Users\ghate\AppData\Local\Programs\Python\Python313\Lib\site-packages (3.1.5)




[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ghate\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
# Dataset 1 : online retail
retail = pd.read_excel('online_retail_II.xlsx')
retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
retail.shape

(525461, 8)

In [5]:
retail.describe


<bound method NDFrame.describe of        Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
525456  538171     22271                 FELTCRAFT DOLL ROSIE         2   
525457  538171     22750         FELTCRAFT PRINCESS LOLA DOLL         1   
525458  538171     22751       FELTCRAFT PRINCESS OLIVIA DOLL         1   
525459  538171     20970   PINK FLORAL FELTCRAFT SHOULDER BAG         2   
525460  538171     21931               JUMBO STORAGE BAG SUKI         2   

               InvoiceDate  Price  Customer ID         Country  


In [6]:
print(retail.columns.tolist())
print(retail.dtypes)

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object


In [7]:
retail['is_cancelled'] = retail['Invoice'].astype(str).str.startswith('C').astype(int)
retail['is_cancelled'].value_counts()

is_cancelled
0    515255
1     10206
Name: count, dtype: int64

In [8]:
retail.isnull().sum()

Invoice              0
StockCode            0
Description       2928
Quantity             0
InvoiceDate          0
Price                0
Customer ID     107927
Country              0
is_cancelled         0
dtype: int64

In [9]:
retail = retail.drop('Customer ID', axis=1, errors='ignore')

In [10]:
retail.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Country', 'is_cancelled'],
      dtype='object')

In [11]:
retail = retail.drop('Description', axis=1, errors ='ignore')

In [12]:
retail.columns

Index(['Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Price', 'Country',
       'is_cancelled'],
      dtype='object')

In [13]:
# Extracting month, day and time from Invoicedate
retail['month']= retail['InvoiceDate'].dt.month
retail['day_of_week_num']=retail['InvoiceDate'].dt.dayofweek
retail['hour']=retail['InvoiceDate'].dt.hour

In [14]:
retail[['month','day_of_week_num','hour']].describe()

,month,day_of_week_num,hour
count,525461.000000,525461.000000,525461.000000
mean,7.438630,2.495709,12.987327
std,3.543294,1.916019,2.426283
min,1.000000,0.000000,7.000000
25%,4.000000,1.000000,11.000000
50%,8.000000,2.000000,13.000000
75%,11.000000,4.000000,15.000000
max,12.000000,6.000000,21.000000


In [15]:
retail['day_of_week_num'].value_counts()

day_of_week_num
3    99292
1    94818
0    93973
2    90486
6    74881
4    71609
5      402
Name: count, dtype: int64

In [16]:
#checking outliers
retail[['Quantity','Price']].describe()    #negative price = outlier, -ve quantity = cancellations

,Quantity,Price
count,525461.000000,525461.000000
mean,10.337667,4.688834
std,107.424110,146.126914
min,-9600.000000,-53594.360000
25%,1.000000,1.250000
50%,3.000000,2.100000
75%,10.000000,4.210000
max,19152.000000,25111.090000


In [17]:
retail[retail['Price']<0][['Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Price', 'Country','is_cancelled']].head()

,Invoice,StockCode,Quantity,InvoiceDate,Price,Country,is_cancelled
179403,A506401,B,1,2010-04-29 13:36:00,-53594.36,United Kingdom,0
276274,A516228,B,1,2010-07-19 11:24:00,-44031.79,United Kingdom,0
403472,A528059,B,1,2010-10-20 12:04:00,-38925.87,United Kingdom,0


In [18]:
(retail['StockCode'] == 'B').sum()

np.int64(3)

In [19]:
retail=retail[retail['StockCode']!='B']

In [20]:
retail[['Quantity','Price']].describe()

,Quantity,Price
count,525458.000000,525458.000000
mean,10.337721,4.948734
std,107.424415,96.493161
min,-9600.000000,0.000000
25%,1.000000,1.250000
50%,3.000000,2.100000
75%,10.000000,4.210000
max,19152.000000,25111.090000


In [21]:
(retail['Price']==0).sum()

np.int64(3687)

In [22]:
retail[retail['Price'] == 0].head(10)

,Invoice,StockCode,Quantity,InvoiceDate,Price,Country,is_cancelled,month,day_of_week_num,hour
263,489464,21733,-96,2009-12-01 10:52:00,0.0,United Kingdom,0,12,1,10
283,489463,71477,-240,2009-12-01 10:52:00,0.0,United Kingdom,0,12,1,10
284,489467,85123A,-192,2009-12-01 10:53:00,0.0,United Kingdom,0,12,1,10
470,489521,21646,-50,2009-12-01 11:44:00,0.0,United Kingdom,0,12,1,11
3114,489655,20683,-44,2009-12-01 17:26:00,0.0,United Kingdom,0,12,1,17
3161,489659,21350,230,2009-12-01 17:39:00,0.0,United Kingdom,0,12,1,17
3162,489660,35956,-1043,2009-12-01 17:43:00,0.0,United Kingdom,0,12,1,17
3168,489663,35605A,-117,2009-12-01 18:02:00,0.0,United Kingdom,0,12,1,18
3731,489781,84292,17,2009-12-02 11:45:00,0.0,United Kingdom,0,12,2,11
4296,489806,18010,-770,2009-12-02 12:42:00,0.0,United Kingdom,0,12,2,12


In [23]:
retail= retail[retail['Price']!=0]

In [24]:
retail[retail['Price'] == 0] # removed negative quantity and price=0

,Invoice,StockCode,Quantity,InvoiceDate,Price,Country,is_cancelled,month,day_of_week_num,hour


In [25]:

retail[retail['Quantity'] < 0]['is_cancelled'].value_counts()

is_cancelled
1    10205
Name: count, dtype: int64

In [26]:
retail['Quantity'] = retail['Quantity'].abs() # converting to absolute values

In [27]:
retail['Quantity'].describe()

count    521771.000000
mean         11.586209
std          90.721092
min           1.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       19152.000000
Name: Quantity, dtype: float64

In [28]:
retail['Country'].nunique()
retail['Country'].value_counts()

Country
United Kingdom          482172
EIRE                      9669
Germany                   8122
France                    5772
Netherlands               2768
Spain                     1278
Switzerland               1187
Portugal                  1101
Belgium                   1053
Channel Islands            906
Sweden                     902
Italy                      731
Australia                  654
Cyprus                     554
Austria                    537
Greece                     517
United Arab Emirates       432
Denmark                    428
Norway                     369
Finland                    354
Unspecified                310
USA                        244
Japan                      224
Poland                     194
Malta                      172
Lithuania                  154
Singapore                  117
RSA                        111
Bahrain                    107
Canada                      77
Thailand                    76
Hong Kong                   76


In [29]:
top_countries = retail['Country'].value_counts().nlargest(6).index
retail['Country_grouped'] = retail['Country'].where(retail['Country'].isin(top_countries), 'Other')
retail['Country_grouped'].value_counts()

Country_grouped
United Kingdom    482172
Other              11990
EIRE                9669
Germany             8122
France              5772
Netherlands         2768
Spain               1278
Name: count, dtype: int64

In [30]:
retail = pd.get_dummies(retail, columns=['Country_grouped'], drop_first=True)

In [31]:
retail = retail.drop('Country', axis=1)

In [32]:
retail.columns

Index(['Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Price',
       'is_cancelled', 'month', 'day_of_week_num', 'hour',
       'Country_grouped_France', 'Country_grouped_Germany',
       'Country_grouped_Netherlands', 'Country_grouped_Other',
       'Country_grouped_Spain', 'Country_grouped_United Kingdom'],
      dtype='object')

In [33]:
retail = retail.drop(['Invoice', 'StockCode', 'InvoiceDate'], axis=1)
retail.columns

Index(['Quantity', 'Price', 'is_cancelled', 'month', 'day_of_week_num', 'hour',
       'Country_grouped_France', 'Country_grouped_Germany',
       'Country_grouped_Netherlands', 'Country_grouped_Other',
       'Country_grouped_Spain', 'Country_grouped_United Kingdom'],
      dtype='object')

In [34]:
X = retail.drop('is_cancelled', axis=1)
y = retail['is_cancelled']

X.dtypes

Quantity                            int64
Price                             float64
month                               int32
day_of_week_num                     int32
hour                                int32
Country_grouped_France               bool
Country_grouped_Germany              bool
Country_grouped_Netherlands          bool
Country_grouped_Other                bool
Country_grouped_Spain                bool
Country_grouped_United Kingdom       bool
dtype: object

In [35]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [36]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

is_cancelled
0    0.980442
1    0.019558
Name: proportion, dtype: float64

In [37]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [38]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99    102314
           1       0.37      0.22      0.27      2041

    accuracy                           0.98    104355
   macro avg       0.68      0.61      0.63    104355
weighted avg       0.97      0.98      0.97    104355

[[101550    764]
 [  1597    444]]


In [39]:
clf_balanced = DecisionTreeClassifier(random_state=42, class_weight='balanced')
clf_balanced.fit(X_train, y_train)
y_pred_balanced = clf_balanced.predict(X_test)
print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97    102314
           1       0.12      0.38      0.19      2041

    accuracy                           0.93    104355
   macro avg       0.56      0.66      0.58    104355
weighted avg       0.97      0.93      0.95    104355



In [40]:
# Dataset 2 - Walmart
walmart = pd.read_csv('Walmart.csv')
walmart.head()
 

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [41]:
walmart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [42]:
walmart.describe()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
std,12.988182,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885
min,1.000000,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000


In [43]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
walmart_scaled = walmart.copy()
walmart_scaled[numeric_features] = scaler.fit_transform(walmart[numeric_features])
walmart_scaled[numeric_features].describe()

,Temperature,Fuel_Price,CPI,Unemployment
count,6.435000e+03,6.435000e+03,6.435000e+03,6.435000e+03
mean,1.280854e-16,-1.077684e-15,-7.596789e-16,-4.593408e-16
std,1.000078e+00,1.000078e+00,1.000078e+00,1.000078e+00
min,-3.400861e+00,-1.931672e+00,-1.156548e+00,-2.196548e+00
25%,-7.159044e-01,-9.272803e-01,-1.012445e+00,-5.907810e-01
50%,1.087764e-01,1.882269e-01,2.804854e-01,-6.672093e-02
75%,7.740514e-01,8.200572e-01,1.046025e+00,3.320552e-01
max,2.140386e+00,2.417063e+00,1.414212e+00,3.366059e+00


In [44]:
X = walmart_scaled[['Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']]
y = walmart_scaled['Weekly_Sales']

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [46]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"effective_metric_ effective_metric_: str or callableThe distance metric to use. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
"effective_metric_params_ effective_metric_params_: dictAdditional keyword arguments for the metric function. For most metricswill be same with `metric_params` parameter, but may also contain the`p` parameter value if the `effective_metric_` attribute is set to'minkowski'.",dict,{}


In [47]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = knn.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

MAE: 446539.87
RMSE: 558769.07
R²: 0.0308
